In [1]:
# modules
import duckdb as db
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats

In [2]:
# Loading data

apv = db.query("SELECT * FROM 'apv.parquet'").df()

mrcl = db.query("SELECT * FROM 'risk_class.parquet'").df()

In [92]:
#Validation with field exams

field_exam_validation = db.query(f"""
    WITH raw_matches AS (
        SELECT 
            m.bovine_id,
            m.farm_id,
            m.risk_class,
            m.last_contact,
            m.vig_date,
            a.round_start,
            a.total_pos,
            a.sampling_confidence,
            ABS(date_diff('day', CAST(m.vig_date AS DATE), CAST(a.round_start AS DATE))) as days_diff,
            
            CASE 
              
                WHEN a.total_pos > 0 
                     AND CAST(a.round_start AS DATE) <= CAST(m.vig_date AS DATE)
                     AND ABS(date_diff('day', CAST(m.vig_date AS DATE), CAST(a.round_start AS DATE))) <= 1095
                     THEN 'before_abattoir'
                

                WHEN a.total_pos > 0 
                     AND CAST(a.round_start AS DATE) > CAST(m.vig_date AS DATE)
                     AND ABS(date_diff('day', CAST(m.vig_date AS DATE), CAST(a.round_start AS DATE))) <= 1095
                     THEN 'after_abattoir'

  
                WHEN a.total_pos = 0 
                     AND CAST(a.round_start AS DATE) >= CAST(m.vig_date AS DATE)
                     AND ABS(date_diff('day', CAST(m.vig_date AS DATE), CAST(a.round_start AS DATE))) <= 1095
                     AND a.sampling_confidence IN ('Gold Standard', 'Acceptable')
                     THEN 'after_abattoir_negative'
                
                ELSE 'not validated'
            END AS validation_type
        FROM mrcl m
        LEFT JOIN apv a ON m.farm_id = a.farm_id
    ),
    hierarchical_ranking AS (
        SELECT *,
            CASE 
                WHEN total_pos > 0 THEN 'Positive'
                WHEN total_pos = 0 AND sampling_confidence IN ('Gold Standard', 'Acceptable') THEN 'Negative'
                WHEN total_pos = 0 AND sampling_confidence NOT IN ('Gold Standard', 'Acceptable') THEN 'Under Sampled'
                ELSE 'Absence of exams'
            END AS farm_field_status,
            ROW_NUMBER() OVER (

                PARTITION BY bovine_id, farm_id 
                ORDER BY 
                    CASE WHEN validation_type = 'before_abattoir' THEN 0 
                         WHEN validation_type = 'after_abattoir' THEN 1  
                         WHEN validation_type = 'after_abattoir_negative' THEN 2 
                         WHEN validation_type = 'not validated' AND sampling_confidence IS NOT NULL THEN 3
                         ELSE 4 END,
                    days_diff ASC
            ) as rank
        FROM raw_matches
    )
    SELECT 
        * EXCLUDE (rank, days_diff) 
    FROM hierarchical_ranking 
    WHERE rank = 1
""").df()

# Preenchimento de nulos para manter a integridade dos 1452 contatos
field_exam_validation['farm_field_status'] = field_exam_validation['farm_field_status'].fillna('Absence of exams')
field_exam_validation['validation_type'] = field_exam_validation['validation_type'].fillna('not validated')

In [93]:
# How many routes with at least one validated exam?

print(field_exam_validation.bovine_id.unique().shape[0])
print(field_exam_validation[field_exam_validation.validation_type != 'not validated'].bovine_id.unique().shape[0])

502
453


In [94]:
# How many contacts are validated?

print(field_exam_validation.farm_id.count())
print(field_exam_validation[field_exam_validation.validation_type != 'not validated'].farm_id.count())

1452
728


In [95]:
# Model Metrics

def get_rr_wilson_score(p1, n1, p2, n2, conf=0.95):
    if n1 == 0 or n2 == 0 or p1 == 0:
        return 1.0, (np.nan, np.nan)
    
    rr = p2 / p1
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    
    l1 = (2 * n1 * p1 + z**2 - z * np.sqrt(z**2 + 4 * n1 * p1 * (1 - p1))) / (2 * (n1 + z**2))
    u1 = (2 * n1 * p1 + z**2 + z * np.sqrt(z**2 + 4 * n1 * p1 * (1 - p1))) / (2 * (n1 + z**2))
    l2 = (2 * n2 * p2 + z**2 - z * np.sqrt(z**2 + 4 * n2 * p2 * (1 - p2))) / (2 * (n2 + z**2))
    u2 = (2 * n2 * p2 + z**2 + z * np.sqrt(z**2 + 4 * n2 * p2 * (1 - p2))) / (2 * (n2 + z**2))
    
    low_limit = rr * np.exp(-z * np.sqrt((1 - l1) / (n1 * l1) + (1 - u2) / (n2 * u2)))
    high_limit = rr * np.exp(z * np.sqrt((1 - u1) / (n1 * u1) + (1 - l2) / (n2 * l2)))
    
    return rr, (low_limit, high_limit)

df_val = field_exam_validation[field_exam_validation['validation_type'] != 'not validated'].copy()
success_bovines = df_val[df_val['farm_field_status'] == 'Positive']['bovine_id'].unique()
df_success = df_val[df_val['bovine_id'].isin(success_bovines)].copy()

risk_classes = ['High', 'Medium', 'Low']

def get_counts(df_sub):
    pos = (df_sub['farm_field_status'] == 'Positive').sum()
    total = (df_sub['farm_field_status'].isin(['Positive', 'Negative'])).sum()
    p = pos / total if total > 0 else 0
    return pos, total, p

stats_all = {r: get_counts(df_val[df_val['risk_class'] == r]) for r in risk_classes}
stats_suc = {r: get_counts(df_success[df_success['risk_class'] == r]) for r in risk_classes}

results = []
for r in risk_classes:
    p_all, n_all = stats_all[r][2], stats_all[r][1]
    p_suc, n_suc = stats_suc[r][2], stats_suc[r][1]
    
    rr_all, ci_all = get_rr_wilson_score(stats_all['Low'][2], stats_all['Low'][1], p_all, n_all)
    rr_suc, ci_suc = get_rr_wilson_score(stats_suc['Low'][2], stats_suc['Low'][1], p_suc, n_suc)
    
    results.append({
        'Risk Class': r,
        'Validated N': n_all,
        'PPV (%)': round(p_all * 100, 2),
        'NPV (%)': round((1 - p_all) * 100, 2),
        'RR (Global)': f"{rr_all:.2f} [{ci_all[0]:.2f} - {ci_all[1]:.2f}]" if r != 'Low' else "1.00 (Ref)",
        'PPV (Surv. Success) (%)': round(p_suc * 100, 2),
        'NPV (Surv. Success) (%)': round((1 - p_suc) * 100, 2),
        'RR (Surv. Success)': f"{rr_suc:.2f} [{ci_suc[0]:.2f} - {ci_suc[1]:.2f}]" if r != 'Low' else "1.00 (Ref)"
    })

p_g_all, n_g_all = get_counts(df_val)[2], get_counts(df_val)[1]
p_g_suc, n_g_suc = get_counts(df_success)[2], get_counts(df_success)[1]

results.append({
    'Risk Class': 'Surveillance (Global)',
    'Validated N': n_g_all,
    'PPV (%)': round(p_g_all * 100, 2),
    'NPV (%)': round((1 - p_g_all) * 100, 2),
    'RR (Global)': '-',
    'PPV (Surv. Success) (%)': round(p_g_suc * 100, 2),
    'NPV (Surv. Success) (%)': round((1 - p_g_suc) * 100, 2),
    'RR (Surv. Success)': '-'
})

metrics_matrix = pd.DataFrame(results).set_index('Risk Class')
display(metrics_matrix)

,Validated N,PPV (%),NPV (%),RR (Global),PPV (Surv. Success) (%),NPV (Surv. Success) (%),RR (Surv. Success)
Risk Class,,,,,,,
High,466,49.36,50.64,2.94 [1.76 - 4.09],80.99,19.01,2.07 [1.31 - 2.70]
Medium,143,23.08,76.92,1.37 [0.78 - 2.22],51.56,48.44,1.31 [0.81 - 1.96]
Low,119,16.81,83.19,1.00 (Ref),39.22,60.78,1.00 (Ref)
Surveillance (Global),728,38.87,61.13,-,70.93,29.07,-
